Step 1:
Tokenizing the-verdit.txt file

In [1]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of characters: ", len(raw_text))
print(raw_text[:10])

Total number of characters:  20479
I HAD alwa


The Goal is to tokenize 20479-character short story into individual words and special characters, these can later be converted to embeddings for LLM training

Regular expressions are used to split and work on text. To preprocess/tokenize

In [2]:
import re

text = "Hello, world. This is a test."
output = re.split(r'(\s)', text)
print(output)

['Hello,', ' ', 'world.', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test.']


In [3]:
output = re.split(r'([,.]|\s)', text)
print(output)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [4]:
output = [item for item in output if item.strip()]
print(output)

['Hello', ',', 'world', '.', 'This', 'is', 'a', 'test', '.']


Keeping or removing whitespaces depends on the use case. For codes where spaces matter its not needed to remove spaces.

In [5]:
# Tokenization scheme as text data has --, ? and other characters that are important

text = "hello, world. Is this-- a test?"
output = re.split(r'([,.-?!_:;()\'"]|--|\s)',text)
output = [ item for item in output if item.strip()]
print(output)

['hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [6]:
# Applying tokenization rules to the raw text data
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [ item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:100])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence', '.', ')', '"', 'The', 'height', 'of', 'his', 'glory', '"', '--', 'that', 'was', 'what', 'the', 'women', 'called', 'it', '.', 'I', 'can', 'hear', 'Mrs', '.', 'Gideon', 'Thwing', '--', 'his', 'last', 'Chicago', 'sitter', '--']


In [7]:
print(len(preprocessed))

4690


Step 2:
Converting tokens to token IDs

Each Unique token will be mapped to a unique integer called tokenID

In [8]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [9]:
# Creating a dictionary with tokens mapped to their IDs
vocab =  { token:integer for integer,token in enumerate(all_words)}

In [10]:
for i,item in enumerate(vocab.items()):
    print(item)
    if(i>=50):
        break


('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [27]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    # Converts sample text to token ids
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.split()
        ]

        ids = [self.str_to_int[s] for s in preprocessed ]
        return ids
    
    # Convering token IDs to their respective tokens and then recreating sentences
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])

        #Replacing spaces before the specified punctutaions
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [ ]:
tok = SimpleTokenizerV1(vocab)

text = """It's the last he painted, you know," Mrs.Gisburn said with pardonable pride."""

ids= tok.encode(text)
print(ids)

[56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [17]:
dec = tok.decode(ids)
print(dec)

It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [ ]:
#text = "Hello, do you like tea?"
#ids = tok.encode(text)
#print(ids)

The example above tells us how the tokenizer reacts to unknown text. It throws an error.

Speical Context Tokens are used in LLMs to prevent such errors.

Now a special case function will be implemented to handle unknown text.
1. After enumerating every word in the vocabulary with unique token ID, an extra token ID will be created for handling all unknown words.
2. END OF TEXT token - When adding multiple data sources. eg: 2 books, in this case after tokenizing the first book, an END OF TEXT token will be inserted to break. 
3. END OF TEXT token will usually be added at the start of new text source. This leads to effective processing of text by LLM
END OF TEXT Token was used when GPT was trained.

In [29]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>","<|unk|>"])

vocab = { token:integer for integer, token in enumerate(all_tokens)}
len(vocab)

1132

In [30]:
# Verifying the presence of additional tokens

for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [31]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()} 
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [ item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        
        ids = [ self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replacing spaces before specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [33]:
tok2 = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

# Adding END OF TEXT token when joining two text sources before passing it to the encoder.
text = " <|endoftext|> ".join((text1,text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [34]:
ids = tok2.encode(text)
print(ids)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [36]:
text = tok2.decode(ids)
print(text)

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


The END OF TEXT token and unknown word token are successfully applied, when decoding they will comeout as initially set in the vocab

BOS - Beginning of sequence
EOS - End of sequence; Useful when combining two sources.
PAD - Padding token; When working with mulitple text sources, if one batch has larger text than the other, the shorter text is padded/extended upto the length of the longest text.

GPT doesnt use unknown token for unknown words but they do use end of text token for simplicity.

GPT models use byte pair encoding tokenizer - They break down words into subword units.  